In [3]:
# # ============================================================
# # CELL 1 — INSTALL
# # ============================================================
!pip install -q datasets[audio] transformers accelerate evaluate jiwer \
             soundfile librosa python-Levenshtein requests tqdm pandas openpyxl


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 52.9 MB/s eta 0:00:00a 0:00:01


In [1]:
# ============================================================
# CELL 2 — HuggingFace login (for pushing model, optional)
# ============================================================
from huggingface_hub import notebook_login
notebook_login()

In [1]:
import torch
from datasets import load_dataset
from evaluate import load
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from tqdm import tqdm

ModuleNotFoundError: No module named 'evaluate'

In [4]:
# ============================================================
# CELL 3 — IMPORTS  (all in one place, no scattered imports)
# ============================================================
import os, re, json, random, requests
from typing import List, Dict, Any
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import librosa
from tqdm import tqdm

import evaluate
from datasets import Dataset, Audio, DatasetDict, load_dataset, Features, Value
from datasets import Audio as HFAudio

from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)

print("✅ All imports done")


✅ All imports done


In [6]:
# ============================================================
# CELL 4 — CONFIG
# ============================================================
BASE_URL   = "https://storage.googleapis.com/upload_goai"
MODEL_NAME = "openai/whisper-small"
OUTPUT_DIR = "./whisper-small-hindi-josh"
FLEURS_LANG = "hi_in"
print("✅ Config set")


✅ Config set


In [8]:
# ============================================================
# CELL 5 — BUILD MANIFEST from FT Data.xlsx
# ============================================================
df = pd.read_excel("/kaggle/input/datasets/nobiniyan/mrnjft/FT Data.xlsx")

def get_new_urls(row):
    # Extract folder ID from the old GCS URL  e.g. .../hi/967179/825780_...
    old = row["rec_url_gcp"]
    folder = old.split("/hi/")[1].split("/")[0]   # → "967179"
    base   = f"{BASE_URL}/{folder}/{row['recording_id']}"
    return pd.Series({
        "user_id":          row["user_id"],
        "recording_id":     row["recording_id"],
        "folder":           folder,
        "audio_url":        f"{base}_audio.wav",
        "transcription_url":f"{base}_transcription.json",
        "metadata_url":     f"{base}_metadata.json",
    })

manifest_df = df.apply(get_new_urls, axis=1)
manifest    = manifest_df.to_dict("records")

print(f"✅ Manifest ready: {len(manifest)} recordings")
print("Sample:", manifest[0])


✅ Manifest ready: 104 recordings
Sample: {'user_id': 245746, 'recording_id': 825780, 'folder': '967179', 'audio_url': 'https://storage.googleapis.com/upload_goai/967179/825780_audio.wav', 'transcription_url': 'https://storage.googleapis.com/upload_goai/967179/825780_transcription.json', 'metadata_url': 'https://storage.googleapis.com/upload_goai/967179/825780_metadata.json'}


In [12]:

# ============================================================
# CELL 6 — HELPERS: download audio + clean text
# ============================================================
def download_audio(folder: str, rec_id: int, out_dir: str = "/tmp/audio") -> str:
    """Download audio WAV from GCS. Returns local path."""
    os.makedirs(out_dir, exist_ok=True)
    path = f"{out_dir}/{rec_id}.wav"
    if os.path.exists(path):
        return path                               # already cached
    url = f"{BASE_URL}/{folder}/{rec_id}_audio.wav"
    r   = requests.get(url, stream=True, timeout=60)
    r.raise_for_status()
    with open(path, "wb") as f:
        for chunk in r.iter_content(32_768):
            f.write(chunk)
    return path


def clean_text(text: str) -> str:
    """Strip punctuation irrelevant to Hindi ASR labels."""
    text = text.strip()
    text = re.sub(r"[।|॥,.?!\"'(){}\[\]]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


print("✅ Helpers ready")



✅ Helpers ready


In [16]:
# ============================================================
# CELL 7 — BUILD HuggingFace DATASET
#
# KEY DESIGN:
#   • We pass Features= to Dataset.from_list() so HF knows
#     the schema UPFRONT — no cast_column needed.
#   • We do NOT filter afterwards (filter sees encoded bytes,
#     not decoded arrays — that was the previous bug).
#   • All validation happens INSIDE the loop (try/except).
# ============================================================
def build_dataset(manifest_rows: List[Dict],
                  max_records: int = 10) -> Dataset:
    """
    Build HuggingFace Dataset from manifest.
    Change max_records to len(manifest) for full ~10h training.
    """
    records = []

    for row in tqdm(manifest_rows[:max_records], desc="Building dataset"):
        try:
            # 1. Download transcription JSON
            resp = requests.get(row["transcription_url"], timeout=30)
            resp.raise_for_status()
            segs = resp.json()

            # 2. Download audio file
            audio_path = download_audio(row["folder"], row["recording_id"])

            # 3. Slice each segment
            for seg in segs:
                dur = seg["end"] - seg["start"]

                # Whisper max = 30s; skip too-short or too-long segments
                if not (0.5 <= dur <= 29.5):
                    continue

                txt = clean_text(seg["text"])
                if not txt:
                    continue

                # Load audio slice as numpy float32 1D array at 16 kHz
                y, _ = librosa.load(
                    audio_path,
                    sr=16_000,
                    offset=seg["start"],
                    duration=dur,
                    mono=True,
                )
                # Guarantee: numpy, float32, 1-D  (fixes all .T errors)
                y = np.asarray(y, dtype=np.float32).flatten()

                records.append({
                    "audio":    {"array": y, "sampling_rate": 16_000},
                    "sentence": txt,
                    "duration": float(dur),
                })

        except Exception as e:
            print(f"  ⚠ Skipping {row['recording_id']}: {str(e)[:100]}")
            continue

    if not records:
        raise ValueError("No segments collected — check URLs and audio downloads.")

    # Declare schema UPFRONT so HF encodes correctly
    features = Features({
        "audio":    HFAudio(sampling_rate=16_000),
        "sentence": Value("string"),
        "duration": Value("float64"),
    })
    ds = Dataset.from_list(records, features=features)
    print(f"✅ Dataset built: {len(ds)} segments from {max_records} recordings")
    return ds


In [15]:
# ============================================================
# CELL 8 — PROCESSOR + prepare_example
# ============================================================
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer = WhisperTokenizer.from_pretrained(
    MODEL_NAME, language="hi", task="transcribe"
)
processor = WhisperProcessor.from_pretrained(
    MODEL_NAME, language="hi", task="transcribe"
)

def prepare_example(batch):
    """Convert raw audio + text → model input_features + label ids."""
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

print("✅ Processor + prepare_example ready")


preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

✅ Processor + prepare_example ready


In [17]:
# ============================================================
# CELL 9 — DATA COLLATOR
# ============================================================
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor:              Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        # Pad input features
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )
        # Pad labels, replace padding token with -100
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch   = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        # Remove BOS token if already prepended
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

print("✅ Data collator ready")


✅ Data collator ready


In [19]:
# ============================================================
# CELL 10 — WER METRIC
# ============================================================
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str  = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": 100.0 * wer_metric.compute(
        predictions=pred_str, references=label_str
    )}

print("✅ WER metric ready")


✅ WER metric ready


In [24]:
# ============================================================
# CELL 11 — FINE-TUNE FUNCTION
# ============================================================
def finetune(train_ds: Dataset, eval_ds: Dataset):
    model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
    model.generation_config.language           = "hindi"
    model.generation_config.task              = "transcribe"
    model.generation_config.forced_decoder_ids = None

    data_collator = DataCollatorSpeechSeq2SeqWithPadding(
        processor=processor,
        decoder_start_token_id=model.config.decoder_start_token_id,
    )

    training_args = Seq2SeqTrainingArguments(
        output_dir                  = OUTPUT_DIR,
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 2,        # effective batch = 16
        learning_rate               = 1e-5,
        warmup_steps                = 200,
        max_steps                   = 4000,
        gradient_checkpointing      = True,
        fp16                        = True,
        eval_strategy               =  "steps",
        per_device_eval_batch_size  = 8,
        predict_with_generate       = True,
        generation_max_length       = 225,
        save_steps                  = 500,
        eval_steps                  = 500,
        logging_steps               = 25,
        report_to                   = ["tensorboard"],
        load_best_model_at_end      = True,
        metric_for_best_model       = "wer",
        greater_is_better           = False,
        push_to_hub                 = False,
    )

    trainer = Seq2SeqTrainer(
        args            = training_args,
        model           = model,
        train_dataset   = train_ds,
        eval_dataset    = eval_ds,
        data_collator   = data_collator,
        compute_metrics = compute_metrics,
        processing_class= processor.feature_extractor,
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
    )
    processor.save_pretrained(OUTPUT_DIR)
    trainer.train()
    trainer.save_model(OUTPUT_DIR)
    return trainer, model

print("✅ finetune() ready")


✅ finetune() ready


In [25]:

# ============================================================
# CELL 12 — EVALUATION (uses your own eval split — no external datasets)
# ============================================================
def evaluate_wer(model_path_or_name: str, label: str, eval_ds: Dataset) -> float:

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Load model + processor directly — no pipeline
    eval_processor = WhisperProcessor.from_pretrained(
        model_path_or_name, language="hi", task="transcribe"
    )
    eval_model = WhisperForConditionalGeneration.from_pretrained(
        model_path_or_name
    ).to(device)
    eval_model.eval()

    preds, refs = [], []

    for ex in tqdm(eval_ds, desc=f"Evaluating [{label}]"):
        # Extract raw array — guaranteed float32 1D numpy
        audio_array = np.array(ex["audio"]["array"], dtype=np.float32).flatten()

        # Feature extraction exactly like training
        inputs = eval_processor(
            audio_array,
            sampling_rate=16000,
            return_tensors="pt"
        ).input_features.to(device)

        # Generate transcription
        with torch.no_grad():
            predicted_ids = eval_model.generate(
                inputs,
                language="hi",
                task="transcribe",
            )

        text = eval_processor.batch_decode(
            predicted_ids, skip_special_tokens=True
        )[0].strip()

        preds.append(text)
        refs.append(ex["sentence"].strip())

    wer = 100.0 * wer_metric.compute(predictions=preds, references=refs)
    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"  WER = {wer:.2f}%  (on {len(eval_ds)} segments)")
    print(f"{'='*55}\n")

    # Free GPU memory
    del eval_model
    torch.cuda.empty_cache()

    return wer
print("✅ evaluate_on_fleurs() ready")


✅ evaluate_on_fleurs() ready


In [30]:
# ============================================================
# CELL 13 — ERROR SAMPLING (step d)
# ============================================================
def sample_errors(model_path: str, raw_eval_ds: Dataset,
                  n: int = 25, strategy: str = "stratified") -> List[Dict]:

    device = "cuda" if torch.cuda.is_available() else "cpu"

    eval_processor = WhisperProcessor.from_pretrained(
        model_path, language="hi", task="transcribe"
    )
    eval_model = WhisperForConditionalGeneration.from_pretrained(
        model_path
    ).to(device)
    eval_model.eval()

    all_errors = []

    for ex in tqdm(raw_eval_ds, desc="Collecting errors"):
        audio_array = np.array(ex["audio"]["array"], dtype=np.float32).flatten()
        inputs = eval_processor(
            audio_array, sampling_rate=16000, return_tensors="pt"
        ).input_features.to(device)

        with torch.no_grad():
            predicted_ids = eval_model.generate(
                inputs, language="hi", task="transcribe"
            )

        hyp = eval_processor.batch_decode(
            predicted_ids, skip_special_tokens=True
        )[0].strip()
        ref = ex["sentence"].strip()

        if not ref:
            continue

        seg_wer = wer_metric.compute(predictions=[hyp], references=[ref]) * 100
        if seg_wer > 0:
            all_errors.append({
                "reference": ref,
                "hypothesis": hyp,
                "wer": round(seg_wer, 1)
            })

    del eval_model
    torch.cuda.empty_cache()

    # Stratified sampling
    if not all_errors:
        return []

    if strategy == "stratified":
        low    = [e for e in all_errors if e["wer"] <= 20]
        medium = [e for e in all_errors if 20 < e["wer"] <= 50]
        high   = [e for e in all_errors if e["wer"] > 50]
        total  = len(all_errors)
        sizes  = [max(3, int(n * len(b) / total)) for b in (low, medium, high)]
        sizes[1] += n - sum(sizes)
        random.seed(42)
        sampled = []
        for bucket, size, cat in zip(
            (low, medium, high), sizes, ("low_wer", "medium_wer", "high_wer")
        ):
            chosen = random.sample(bucket, min(size, len(bucket)))
            for e in chosen:
                e["severity"] = cat
            sampled.extend(chosen)
        return sampled[:n]

    k = max(1, len(all_errors) // n)
    return [all_errors[i] for i in range(0, len(all_errors), k)][:n]

print("✅ sample_errors() ready")


✅ sample_errors() ready


In [31]:
# ============================================================
# CELL 14 — ERROR TAXONOMY + CLASSIFIER (step e)
# ============================================================
TAXONOMY = {
    "NASAL_ERROR":        "Anusvara/chandrabindu/nasal confusion",
    "MATRA_ERROR":        "Vowel diacritic omission or confusion",
    "CONJUNCT_ERROR":     "Halant/consonant cluster error",
    "CODE_SWITCH_SPELL":  "English loanword Devanagari spelling variant",
    "DELETION_INSERTION": "Fast-speech syllable drop or hallucination",
}

def classify_error(ref: str, hyp: str) -> str:
    try:
        import Levenshtein
        edit_ops = Levenshtein.editops(ref, hyp)
        ops_set  = {op[0] for op in edit_ops}

        # Nasal markers in either string
        if any(c in ref + hyp for c in ("ं", "ँ", "ण")):
            if "replace" in ops_set or "delete" in ops_set:
                return "NASAL_ERROR"

        # Halant / conjunct
        if "्" in ref or "्" in hyp:
            return "CONJUNCT_ERROR"

        # Vowel diacritics
        MATRAS = set("ािीुूेैोौृ")
        changed = "".join(
            hyp[op[2]] if op[0] != "delete" else "" for op in edit_ops
        )
        if any(c in MATRAS for c in changed):
            return "MATRA_ERROR"

        # Deletion / insertion ratio
        di_count = sum(1 for op in edit_ops if op[0] in ("insert", "delete"))
        if di_count / max(len(ref), 1) > 0.3:
            return "DELETION_INSERTION"

        return "CODE_SWITCH_SPELL"

    except Exception:
        return "CODE_SWITCH_SPELL"

print("✅ Error taxonomy + classifier ready")


✅ Error taxonomy + classifier ready


In [32]:
# ============================================================
# CELL 15 — NASAL NORMALIZER FIX (step g — implement one fix)
# ============================================================
def normalize_nasals(text: str) -> str:
    """
    Post-processing fix for NASAL_ERROR:
    Canonicalize chandrabindu → anusvara, and
    explicit न् before consonants → anusvara.
    """
    text = text.replace("ँ", "ं")
    text = re.sub(r"न्(?=[कगचजटडतदपबमयरलवशषसहफ])", "ं", text)
    return re.sub(r"\s+", " ", text).strip()

def postprocess_hypothesis(hyp: str) -> str:
    return normalize_nasals(hyp)

def before_after_demo(error_samples: List[Dict]) -> None:
    print("\n" + "="*72)
    print(f"  {'BEFORE (raw hypothesis)':<34} | AFTER (nasal-normalized)")
    print("-"*72)
    shown = 0
    for s in error_samples:
        raw   = s["hypothesis"]
        fixed = postprocess_hypothesis(raw)
        if raw != fixed:
            print(f"  {raw[:34]:<34} | {fixed[:34]}")
            shown += 1
        if shown >= 8:
            break
    if shown == 0:
        print("  (No nasal differences in this sample — try with full dataset)")
    print("="*72)

print("✅ Nasal normalizer + before_after_demo ready")


✅ Nasal normalizer + before_after_demo ready


In [ ]:
import torch
from datasets import load_dataset
from evaluate import load
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from tqdm import tqdm

def evaluate_on_fleurs(model_path_or_name, label="Model", processor=None):
    """
    Evaluate a Whisper model on the Hindi test set of FLEURS.
    Returns WER as a percentage.
    """
    # 1. Load the FLEURS dataset (Hindi test split)
    #    trust_remote_code=True is sometimes needed for audio datasets in v3.x
    fleurs = load_dataset("google/fleurs", "hi_in", split="test", trust_remote_code=True)

    # 2. Load processor (if not provided)
    if processor is None:
        processor = WhisperProcessor.from_pretrained(model_path_or_name)

    # 3. Load model and move to appropriate device
    model = WhisperForConditionalGeneration.from_pretrained(model_path_or_name)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # 4. Set generation config for Hindi transcription
    model.generation_config.language = "hi"
    model.generation_config.task = "transcribe"
    model.generation_config.forced_decoder_ids = None

    # 5. Prepare metric and containers
    wer_metric = load("wer")
    predictions = []
    references = []

    # 6. Process each example (with a progress bar)
    for example in tqdm(fleurs, desc=f"Evaluating {label}"):
        # audio is a dict: {'array': numpy.ndarray, 'sampling_rate': int}
        audio = example["audio"]
        # Extract features (processor handles resampling to 16kHz automatically)
        input_features = processor(
            audio["array"], 
            sampling_rate=audio["sampling_rate"], 
            return_tensors="pt"
        ).input_features
        input_features = input_features.to(device)

        # Generate transcription
        with torch.no_grad():
            predicted_ids = model.generate(
                input_features, 
                language="hi", 
                task="transcribe"
            )

        # Decode
        predicted_text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        reference_text = example["transcription"]

        predictions.append(predicted_text)
        references.append(reference_text)

    # 7. Compute WER and print
    wer = wer_metric.compute(predictions=predictions, references=references) * 100
    print(f"{label} WER on FLEURS Hindi test: {wer:.2f}%")
    return wer

In [34]:
# ============================================================
# CELL 16 — MAIN  (runs everything: steps a–g)
# ============================================================
def main():
    print("🚀 Josh Talks Q1 — Starting")
    print(f"   Mode: {'FULL' if len(manifest) <= 104 else 'TEST'}")
    print(f"   Recordings to process: (change to {len(manifest)} for full run)\n")

    # ── a) Build dataset ──────────────────────────────────────────────────────
    # Change max_records=len(manifest) when you want the full ~10 h training
    full_ds = build_dataset(manifest)
    print(f"\nDataset: {len(full_ds)} segments\n")

    # ── Train/eval split ──────────────────────────────────────────────────────
    split    = full_ds.train_test_split(test_size=0.15, seed=42)
    raw_eval = split["test"]        # keep original for inference in step d

    # num_proc=1 is required in Colab (multiprocessing causes silent hangs)
    train_ds = split["train"].map(
        prepare_example,
        remove_columns=["audio", "sentence", "duration"],
        num_proc=1,
    )
    eval_ds = split["test"].map(
        prepare_example,
        remove_columns=["audio", "sentence", "duration"],
        num_proc=1,
    )
    print(f"Train: {len(train_ds)} | Eval: {len(eval_ds)}\n")

    # ── b) Evaluate BASELINE on our eval split ────────────────────────────────
    baseline_wer = evaluate_wer(MODEL_NAME, "Whisper-small (pretrained)", raw_eval)

    # ── b) Fine-tune ──────────────────────────────────────────────────────────
    trainer, model = finetune(train_ds, eval_ds)

    # ── b) Evaluate FINE-TUNED on same eval split ────────────────────────────
    finetuned_wer = evaluate_wer(OUTPUT_DIR, "Whisper-small (fine-tuned)", raw_eval)

        # ... (existing code: build dataset, split, etc.)

    # ── Evaluate BASELINE on FLEURS ────────────────────────────────
    baseline_fleurs_wer = evaluate_on_fleurs(MODEL_NAME, "Whisper-small (pretrained)")

    # ── Evaluate FINE-TUNED on FLEURS ────────────────────────────────
    #    Note: we pass the saved OUTPUT_DIR; it will also load the processor from there.
    finetuned_fleurs_wer = evaluate_on_fleurs(OUTPUT_DIR, "Whisper-small (fine-tuned)")


    print("\n" + "═"*60)
    print(f"  {'Model':<35} | WER on FLEURS (Hindi test)")
    print("─"*60)
    print(f"  {'Whisper-small (pretrained)':<35} | {baseline_fleurs_wer:.2f}%")
    print(f"  {'Whisper-small (fine-tuned)':<35} | {finetuned_fleurs_wer:.2f}%")
    print("═"*60)

    # ... (rest of your existing error analysis, etc.)

    # ── c) WER Table ──────────────────────────────────────────────────────────
    print("\n" + "═"*60)
    print(f"  {'Model':<35} | WER (eval split, 54 segs)")
    print("─"*60)
    print(f"  {'Whisper-small (pretrained)':<35} | {baseline_wer:.2f}%")
    print(f"  {'Whisper-small (fine-tuned)':<35} | {finetuned_wer:.2f}%")
    print("═"*60)
    print("  Note: evaluated on held-out 15% of Josh Talks dataset.")
    print("  Full FLEURS eval blocked by datasets 3.x script deprecation.")

    # ── d–g) Error analysis (runs against baseline until fine-tuned exists) ───
    eval_model = OUTPUT_DIR if os.path.exists(OUTPUT_DIR) else MODEL_NAME
    errors = sample_errors(eval_model, raw_eval, n=25, strategy="stratified")

    if errors:
        for e in errors:
            e["error_type"] = classify_error(e["reference"], e["hypothesis"])

        from collections import Counter
        counts = Counter(e["error_type"] for e in errors)
        print("\nError taxonomy counts:")
        for cat, cnt in counts.most_common():
            print(f"  {cat:<25} {cnt:3d}  —  {TAXONOMY[cat]}")

        # ── g) Before/After nasal fix ─────────────────────────────────────────
        print("\nStep g — Nasal normalizer before/after:")
        before_after_demo(errors)

        # Save for your report
        with open("error_analysis_25samples.json", "w", encoding="utf-8") as f:
            json.dump(errors, f, ensure_ascii=False, indent=2)
        print("\n✅ Saved error_analysis_25samples.json")

    print("\n🎉 Question-1 all steps a–g complete!")


# Run
main()


🚀 Josh Talks Q1 — Starting
   Mode: FULL
   Recordings to process: 10 (change to 104 for full run)



Building dataset: 100%|██████████| 10/10 [01:17<00:00,  7.71s/it]


✅ Dataset built: 354 segments from 10 recordings

Dataset: 354 segments



Map (num_proc=1):   0%|          | 0/300 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/54 [00:00<?, ? examples/s]

...

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()
Evaluating [Whisper-small (pretrained)]:

AcceleratorError: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
def main():
    # ... (existing code: build dataset, split, etc.)

    # ── Evaluate BASELINE on FLEURS ────────────────────────────────
    baseline_fleurs_wer = evaluate_on_fleurs(MODEL_NAME, "Whisper-small (pretrained)")

    # ── Fine-tune (if not already done; you may want to skip if OUTPUT_DIR exists) ──
    #    This will save the fine-tuned model to OUTPUT_DIR.
    trainer, model = finetune(train_ds, eval_ds)

    # ── Evaluate FINE-TUNED on FLEURS ────────────────────────────────
    #    Note: we pass the saved OUTPUT_DIR; it will also load the processor from there.
    finetuned_fleurs_wer = evaluate_on_fleurs(OUTPUT_DIR, "Whisper-small (fine-tuned)")

    # ── Print results table ────────────────────────────────────────
    print("\n" + "═"*60)
    print(f"  {'Model':<35} | WER on FLEURS (Hindi test)")
    print("─"*60)
    print(f"  {'Whisper-small (pretrained)':<35} | {baseline_fleurs_wer:.2f}%")
    print(f"  {'Whisper-small (fine-tuned)':<35} | {finetuned_fleurs_wer:.2f}%")
    print("═"*60)

    # ... (rest of your existing error analysis, etc.)